# 🎯 Ćwiczenie 4: PySpark RDD — Analiza logów serwera WWW

**Kurs:** Python w Zastosowaniach Big Data
**Wykład:** 4 — Apache Spark, architektura, RDD
**Czas trwania:** ⏱️ ~90 minut
**Poziom:** ⭐⭐⭐ (Średni)

## 📚 Cele ćwiczenia

Po ukończeniu tego notebooka będziesz potrafić:
1. Tworzyć RDD z surowych linii tekstu (logów serwera) i parsować je funkcją `map`
2. Filtrować dane (`filter`) i agregować je po kluczu (`reduceByKey`)
3. Wyciągać statystyki biznesowe z ~50 000 wpisów logów: ruch, błędy, opóźnienia, podejrzane IP

## 📋 Wymagania wstępne
- Znajomość podstaw Pythona (listy, słowniki, funkcje lambda)
- Wykład 4: architektura Sparka, RDD, transformacje vs akcje

---
> 💡 **Tip:** Uruchamiaj komórki po kolei (`Shift+Enter`). Jeśli utkniesz — sprawdź komórki z podpowiedziami (💡) albo zapytaj prowadzącego!

In [ ]:
# @title 🔧 Setup — instalacja PySpark i inicjalizacja SparkSession
!pip install -q pyspark

import random
from datetime import datetime, timedelta

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Cwiczenie4-RDD-Logi-Serwera")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

random.seed(42)  # reprodukowalność — zawsze te same "losowe" logi

print("✅ Setup complete! Spark działa, ruszamy 🚀")
print(f"Wersja Sparka: {sc.version}")

## 📖 Quick Refresher — o co chodzi z logami serwera?

Każdy serwer WWW (Apache, Nginx) zapisuje **linię tekstu** dla każdego zapytania HTTP. W tym ćwiczeniu wygenerujemy własne, syntetyczne logi w uproszczonym formacie:

```
<IP> <timestamp> <endpoint> <status_code> <response_time_ms> <user_agent>
```

Przykład:
```
192.168.42.7 2026-10-27T08:15:32 /api/products 200 145 Mozilla/5.0
```

| Pole | Opis | Przykład |
|------|------|---------|
| `ip` | Adres IP klienta | `192.168.42.7` |
| `timestamp` | Data i godzina żądania (ISO 8601) | `2026-10-27T08:15:32` |
| `endpoint` | Ścieżka URL | `/api/products` |
| `status_code` | Kod odpowiedzi HTTP | `200`, `404`, `500` |
| `response_time_ms` | Czas odpowiedzi w milisekundach | `145` |
| `user_agent` | Przeglądarka/bot klienta | `Mozilla/5.0`, `BotCrawler/1.0` |

Będziemy pracować **wyłącznie na RDD** (nie DataFrame!) — bo dziś ćwiczymy `map`, `filter`, `reduceByKey` i inne operacje niskopoziomowe.

In [ ]:
# @title 📊 Generowanie ~50 000 syntetycznych logów serwera

ENDPOINTS = [
    "/home", "/api/products", "/api/products", "/api/cart", "/checkout",
    "/login", "/api/users", "/search", "/static/style.css", "/api/orders",
]
STATUS_CODES = [200] * 80 + [301] * 5 + [404] * 10 + [500] * 3 + [503] * 2
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0)",
    "Mozilla/5.0 (Macintosh)",
    "Mozilla/5.0 (X11; Linux)",
    "BotCrawler/1.0",
]

N_LOGS = 50_000
N_NORMAL_IPS = 1_200
N_BOT_IPS = 5  # kilka IP odpowiedzialnych za nienaturalnie dużo requestów

normal_ips = [f"10.0.{random.randint(0, 255)}.{random.randint(1, 254)}" for _ in range(N_NORMAL_IPS)]
bot_ips = [f"172.16.99.{i}" for i in range(1, N_BOT_IPS + 1)]

start_time = datetime(2026, 10, 27, 0, 0, 0)
log_lines: list[str] = []

for _ in range(N_LOGS):
    # 15% requestów pochodzi z puli "botów" — będą mieć nienaturalnie dużo wpisów
    if random.random() < 0.15:
        ip = random.choice(bot_ips)
        user_agent = "BotCrawler/1.0"
    else:
        ip = random.choice(normal_ips)
        user_agent = random.choice(USER_AGENTS[:-1])

    endpoint = random.choice(ENDPOINTS)
    status = random.choice(STATUS_CODES)
    response_time = max(5, int(random.gauss(150, 80)))
    timestamp = start_time + timedelta(seconds=random.randint(0, 7 * 24 * 3600))

    line = f"{ip} {timestamp.isoformat()} {endpoint} {status} {response_time} {user_agent}"
    log_lines.append(line)

print(f"📊 Wygenerowano {len(log_lines):,} linii logów")
print("Przykładowe linie:")
for line in log_lines[:3]:
    print("  ", line)

# Tworzymy RDD z listy linii tekstowych — dokładnie tak, jak sc.textFile() by to zrobił
logi_rdd = sc.parallelize(log_lines, numSlices=8)
print(f"\n✅ RDD utworzone, liczba partycji: {logi_rdd.getNumPartitions()}")

## 🎓 Przykład prowadzony — parsowanie i filtrowanie logów

Zanim zaczniesz samodzielne ćwiczenia, przeanalizujmy razem, jak sparsować surowe linie na strukturę i odfiltrować błędy.

In [ ]:
# @title 🎓 Przykład prowadzony — parsowanie logów za pomocą map()

def parsuj_linie(linia: str) -> tuple[str, str, str, int, int, str]:
    """Zamienia surową linię logu na krotkę (ip, timestamp, endpoint, status, czas_ms, user_agent)."""
    czesci = linia.split(" ")
    ip, timestamp, endpoint, status, response_time, user_agent = (
        czesci[0], czesci[1], czesci[2], int(czesci[3]), int(czesci[4]), " ".join(czesci[5:])
    )
    return (ip, timestamp, endpoint, status, response_time, user_agent)

# Krok 1: sparsuj każdą linię (transformacja map — nic się jeszcze nie liczy!)
sparsowane_rdd = logi_rdd.map(parsuj_linie)

# Krok 2: odfiltruj tylko błędy (kod 4xx lub 5xx)
bledy_rdd = sparsowane_rdd.filter(lambda rekord: rekord[3] >= 400)

# Krok 3: dopiero take() to akcja — TERAZ Spark faktycznie liczy
przyklad_bledow = bledy_rdd.take(5)
print("Przykładowe błędy (4xx/5xx):")
for rekord in przyklad_bledow:
    print("  ", rekord)

liczba_bledow = bledy_rdd.count()
liczba_wszystkich = sparsowane_rdd.count()
print(f"\n📊 Błędy: {liczba_bledow:,} / {liczba_wszystkich:,} "
      f"({liczba_bledow / liczba_wszystkich:.1%})")

---
## ✏️ Ćwiczenie 1: Top 5 najczęściej odwiedzanych endpointów

**Trudność:** ⭐

Używając `sparsowane_rdd`, policz liczbę wystąpień każdego `endpointu` i zwróć **top 5** najpopularniejszych (malejąco wg liczby wystąpień) jako listę krotek `(endpoint, liczba)`.

**Kroki:**
1. Zamień każdy rekord na parę `(endpoint, 1)` za pomocą `map`
2. Zsumuj wystąpienia po kluczu za pomocą `reduceByKey`
3. Posortuj malejąco i weź 5 pierwszych (`takeOrdered` z ujemnym kluczem, albo `sortBy` + `take`)

> 💡 **Hint:** `reduceByKey(lambda a, b: a + b)` sumuje wartości dla tego samego klucza.

In [ ]:
# ✏️ Ćwiczenie 1: Top 5 endpointów wg ruchu

# TODO: zamień każdy rekord na parę (endpoint, 1)
endpoint_pary_rdd = ...  # sparsowane_rdd.map(lambda rekord: ...)

# TODO: zsumuj wystąpienia po kluczu (endpoint)
endpoint_liczba_rdd = ...  # endpoint_pary_rdd.reduceByKey(...)

# TODO: weź top 5 wg liczby wystąpień malejąco
top5_endpointy = ...  # endpoint_liczba_rdd.takeOrdered(5, key=lambda x: -x[1])

print("Top 5 endpointów wg ruchu:")
for endpoint, liczba in top5_endpointy:
    print(f"  {endpoint}: {liczba:,}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 1 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: zamieniamy każdy sparsowany rekord na parę (endpoint, 1) — przygotowanie do zliczania
endpoint_pary_rdd = sparsowane_rdd.map(lambda rekord: (rekord[2], 1))

# Krok 2: sumujemy jedynki po kluczu (endpoint) — reduceByKey agreguje w obrębie i między partycjami
endpoint_liczba_rdd = endpoint_pary_rdd.reduceByKey(lambda a, b: a + b)

# Krok 3: wybieramy 5 endpointów z najwyższą liczbą wystąpień (ujemny klucz = sortowanie malejąco)
top5_endpointy = endpoint_liczba_rdd.takeOrdered(5, key=lambda x: -x[1])

print("Top 5 endpointów wg ruchu:")
for endpoint, liczba in top5_endpointy:
    print(f"  {endpoint}: {liczba:,}")


In [ ]:
# @title ✅ Test — Ćwiczenie 1
try:
    assert top5_endpointy is not ..., "❌ Nie ukończyłeś/aś zadania"
    assert isinstance(top5_endpointy, list), "❌ Wynik powinien być listą"
    assert len(top5_endpointy) == 5, f"❌ Oczekiwano 5 wyników, jest {len(top5_endpointy)}"
    liczby = [liczba for _, liczba in top5_endpointy]
    assert liczby == sorted(liczby, reverse=True), "❌ Wyniki nie są posortowane malejąco"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie, dasz radę! 💪")

---
## ✏️ Ćwiczenie 2: Średni czas odpowiedzi dla każdego endpointu

**Trudność:** ⭐⭐

Oblicz **średni** `response_time_ms` dla każdego endpointu.

**Podpowiedź strategii:** zamień każdy rekord na `(endpoint, (response_time, 1))`, zsumuj obie wartości przez `reduceByKey`, a na koniec podziel sumę przez licznik w `mapValues`.

> 💡 **Hint:** `reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))` sumuje jednocześnie czas i licznik.

In [ ]:
# ✏️ Ćwiczenie 2: Średni czas odpowiedzi na endpoint

# TODO: zamień na (endpoint, (response_time, 1))
czas_pary_rdd = ...  # sparsowane_rdd.map(lambda rekord: (rekord[2], (rekord[4], 1)))

# TODO: zsumuj (suma_czasu, suma_licznika) po kluczu
czas_suma_rdd = ...  # czas_pary_rdd.reduceByKey(...)

# TODO: policz średnią: suma_czasu / licznik
srednia_czas_rdd = ...  # czas_suma_rdd.mapValues(lambda para: para[0] / para[1])

srednia_czas_wynik = dict(srednia_czas_rdd.collect())
print("Średni czas odpowiedzi (ms) wg endpointu:")
for endpoint, srednia in sorted(srednia_czas_wynik.items(), key=lambda x: -x[1]):
    print(f"  {endpoint}: {srednia:.1f} ms")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 2 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: zamieniamy rekord na (endpoint, (czas_odpowiedzi, 1)) — licznik potrzebny do średniej
czas_pary_rdd = sparsowane_rdd.map(lambda rekord: (rekord[2], (rekord[4], 1)))

# Krok 2: sumujemy jednocześnie czas i licznik dla każdego endpointu
czas_suma_rdd = czas_pary_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Krok 3: dzielimy sumę czasu przez liczbę wystąpień, żeby otrzymać średnią
srednia_czas_rdd = czas_suma_rdd.mapValues(lambda para: para[0] / para[1])

srednia_czas_wynik = dict(srednia_czas_rdd.collect())
print("Średni czas odpowiedzi (ms) wg endpointu:")
for endpoint, srednia in sorted(srednia_czas_wynik.items(), key=lambda x: -x[1]):
    print(f"  {endpoint}: {srednia:.1f} ms")


In [ ]:
# @title ✅ Test — Ćwiczenie 2
try:
    assert srednia_czas_wynik is not ..., "❌ Nie ukończyłeś/aś zadania"
    assert isinstance(srednia_czas_wynik, dict), "❌ Wynik powinien dać się zamienić na słownik"
    assert len(srednia_czas_wynik) == len(set(ENDPOINTS)), "❌ Liczba endpointów się nie zgadza"
    assert all(v > 0 for v in srednia_czas_wynik.values()), "❌ Średnie muszą być dodatnie"
    print("✅ Wszystkie testy przeszły! Great job! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie, dasz radę! 💪")

---
## ✏️ Ćwiczenie 3: Wskaźnik błędów (error rate) na godzinę

**Trudność:** ⭐⭐

Dla każdej **godziny** dnia (0-23, wyciągniętej z `timestamp`) policz **wskaźnik błędów**: liczba requestów z kodem >= 400 podzielona przez liczbę wszystkich requestów w tej godzinie.

**Kroki:**
1. Wyciągnij godzinę z `timestamp` (format `YYYY-MM-DDTHH:MM:SS` — godzina to `timestamp[11:13]`)
2. Zbuduj RDD par `(godzina, (czy_blad, 1))`, gdzie `czy_blad` to `1` jeśli status >= 400, inaczej `0`
3. Zsumuj przez `reduceByKey`, policz wskaźnik = suma_bledow / suma_wszystkich

In [ ]:
# ✏️ Ćwiczenie 3: Error rate na godzinę

# TODO: zamień rekord na (godzina, (czy_blad, 1))
godzina_pary_rdd = ...  # sparsowane_rdd.map(lambda r: (r[1][11:13], (1 if r[3] >= 400 else 0, 1)))

# TODO: zsumuj (suma_bledow, suma_wszystkich) po godzinie
godzina_suma_rdd = ...  # godzina_pary_rdd.reduceByKey(...)

# TODO: policz wskaźnik błędów = suma_bledow / suma_wszystkich
error_rate_rdd = ...  # godzina_suma_rdd.mapValues(lambda para: para[0] / para[1])

error_rate_wynik = dict(error_rate_rdd.collect())
print("Wskaźnik błędów wg godziny (top 5 najgorszych):")
for godzina, rate in sorted(error_rate_wynik.items(), key=lambda x: -x[1])[:5]:
    print(f"  Godzina {godzina}: {rate:.1%}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 3 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: wyciągamy godzinę z timestampu (znaki 11-13) i budujemy parę (godzina, (czy_blad, 1))
godzina_pary_rdd = sparsowane_rdd.map(lambda r: (r[1][11:13], (1 if r[3] >= 400 else 0, 1)))

# Krok 2: sumujemy liczbę błędów i liczbę wszystkich requestów per godzina
godzina_suma_rdd = godzina_pary_rdd.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Krok 3: wskaźnik błędów = liczba błędów / liczba wszystkich requestów w danej godzinie
error_rate_rdd = godzina_suma_rdd.mapValues(lambda para: para[0] / para[1])

error_rate_wynik = dict(error_rate_rdd.collect())
print("Wskaźnik błędów wg godziny (top 5 najgorszych):")
for godzina, rate in sorted(error_rate_wynik.items(), key=lambda x: -x[1])[:5]:
    print(f"  Godzina {godzina}: {rate:.1%}")


In [ ]:
# @title ✅ Test — Ćwiczenie 3
try:
    assert error_rate_wynik is not ..., "❌ Nie ukończyłeś/aś zadania"
    assert isinstance(error_rate_wynik, dict), "❌ Wynik powinien dać się zamienić na słownik"
    assert all(0 <= v <= 1 for v in error_rate_wynik.values()), "❌ Wskaźnik musi być między 0 a 1"
    assert len(error_rate_wynik) <= 24, "❌ Nie może być więcej niż 24 godziny"
    print("✅ Wszystkie testy przeszły! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---
## ✏️ Ćwiczenie 4: 10 najwolniejszych requestów

**Trudność:** ⭐

Znajdź **10 requestów z najdłuższym czasem odpowiedzi** (`response_time_ms`) w całym zbiorze danych, posortowanych malejąco.

> 💡 **Hint:** `takeOrdered(n, key=...)` z ujemnym kluczem sortuje malejąco bez konieczności sortowania całego RDD (`sortBy` + `take`).

In [ ]:
# ✏️ Ćwiczenie 4: 10 najwolniejszych requestów

# TODO: użyj takeOrdered z odpowiednim kluczem sortowania (malejąco po response_time_ms, indeks 4)
najwolniejsze = ...  # sparsowane_rdd.takeOrdered(10, key=lambda rekord: -rekord[4])

print("10 najwolniejszych requestów:")
for rekord in najwolniejsze:
    print(f"  {rekord[2]}: {rekord[4]} ms (IP: {rekord[0]})")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 4 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: takeOrdered z ujemnym kluczem sortuje malejąco po response_time_ms (indeks 4) —
# Spark od razu zbiera tylko 10 najlepszych, bez sortowania całego RDD
najwolniejsze = sparsowane_rdd.takeOrdered(10, key=lambda rekord: -rekord[4])

print("10 najwolniejszych requestów:")
for rekord in najwolniejsze:
    print(f"  {rekord[2]}: {rekord[4]} ms (IP: {rekord[0]})")


In [ ]:
# @title ✅ Test — Ćwiczenie 4
try:
    assert najwolniejsze is not ..., "❌ Nie ukończyłeś/aś zadania"
    assert len(najwolniejsze) == 10, f"❌ Oczekiwano 10 wyników, jest {len(najwolniejsze)}"
    czasy = [rekord[4] for rekord in najwolniejsze]
    assert czasy == sorted(czasy, reverse=True), "❌ Wyniki nie są posortowane malejąco wg czasu"
    print("✅ Wszystkie testy przeszły! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---
## ✏️ Ćwiczenie 5: Liczba requestów per IP — wykrywanie potencjalnych botów

**Trudność:** ⭐⭐

Policz liczbę requestów dla każdego adresu IP i wypisz **top 10 IP** z największą liczbą requestów. Pamiętaj, że w danych celowo umieściliśmy kilka "botowych" adresów IP — powinny być wyraźnie widoczne na szczycie listy!

> 💡 **Hint:** To ten sam wzorzec co w Ćwiczeniu 1, tylko kluczem jest `ip` (indeks 0) zamiast `endpoint`.

In [ ]:
# ✏️ Ćwiczenie 5: Top 10 IP wg liczby requestów

# TODO: zamień każdy rekord na parę (ip, 1)
ip_pary_rdd = ...  # sparsowane_rdd.map(lambda rekord: (rekord[0], 1))

# TODO: zsumuj wystąpienia po IP
ip_liczba_rdd = ...  # ip_pary_rdd.reduceByKey(...)

# TODO: weź top 10 IP wg liczby requestów malejąco
top10_ip = ...  # ip_liczba_rdd.takeOrdered(10, key=lambda x: -x[1])

print("Top 10 IP wg liczby requestów (potencjalne boty na szczycie):")
for ip, liczba in top10_ip:
    print(f"  {ip}: {liczba:,} requestów")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 5 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: zamieniamy każdy rekord na parę (ip, 1) — dokładnie ten sam wzorzec co w Ćwiczeniu 1
ip_pary_rdd = sparsowane_rdd.map(lambda rekord: (rekord[0], 1))

# Krok 2: sumujemy wystąpienia requestów po adresie IP
ip_liczba_rdd = ip_pary_rdd.reduceByKey(lambda a, b: a + b)

# Krok 3: wybieramy 10 adresów IP z największą liczbą requestów — tu powinny być widoczne boty
top10_ip = ip_liczba_rdd.takeOrdered(10, key=lambda x: -x[1])

print("Top 10 IP wg liczby requestów (potencjalne boty na szczycie):")
for ip, liczba in top10_ip:
    print(f"  {ip}: {liczba:,} requestów")


In [ ]:
# @title ✅ Test — Ćwiczenie 5
try:
    assert top10_ip is not ..., "❌ Nie ukończyłeś/aś zadania"
    assert len(top10_ip) == 10, f"❌ Oczekiwano 10 wyników, jest {len(top10_ip)}"
    liczby = [liczba for _, liczba in top10_ip]
    assert liczby == sorted(liczby, reverse=True), "❌ Wyniki nie są posortowane malejąco"
    # znane adresy botów powinny znaleźć się w top 10
    top_ips_set = {ip for ip, _ in top10_ip}
    assert any(ip.startswith("172.16.99.") for ip in top_ips_set), "❌ Boty powinny być widoczne w top 10"
    print("✅ Wszystkie testy przeszły! Boty wykryte! 🕵️🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---
## ✏️ Ćwiczenie 6: Rozkład kodów statusu

**Trudność:** ⭐

Policz, ile razy wystąpił każdy kod statusu HTTP (`200`, `301`, `404`, `500`, `503`) w całym zbiorze danych.

> 💡 **Hint:** Ten sam wzorzec `map` + `reduceByKey` co wcześniej, kluczem jest `status_code` (indeks 3).

In [ ]:
# ✏️ Ćwiczenie 6: Rozkład kodów statusu

# TODO: zamień na (status_code, 1) i zsumuj przez reduceByKey
status_liczba_rdd = ...  # sparsowane_rdd.map(lambda r: (r[3], 1)).reduceByKey(lambda a, b: a + b)

status_wynik = dict(status_liczba_rdd.collect())
print("Rozkład kodów statusu:")
for status, liczba in sorted(status_wynik.items()):
    print(f"  {status}: {liczba:,}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Ćwiczenie 6 (kliknij, aby zobaczyć krok po kroku)

# Krok 1: zamieniamy każdy rekord na (status_code, 1) i sumujemy po kluczu w jednym łańcuchu operacji
status_liczba_rdd = sparsowane_rdd.map(lambda r: (r[3], 1)).reduceByKey(lambda a, b: a + b)

status_wynik = dict(status_liczba_rdd.collect())
print("Rozkład kodów statusu:")
for status, liczba in sorted(status_wynik.items()):
    print(f"  {status}: {liczba:,}")


In [ ]:
# @title ✅ Test — Ćwiczenie 6
try:
    assert status_wynik is not ..., "❌ Nie ukończyłeś/aś zadania"
    assert isinstance(status_wynik, dict), "❌ Wynik powinien dać się zamienić na słownik"
    assert sum(status_wynik.values()) == N_LOGS, "❌ Suma wszystkich statusów powinna równać się liczbie logów"
    assert 200 in status_wynik, "❌ Brakuje kodu 200 w wyniku"
    print("✅ Wszystkie testy przeszły! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj ponownie! 💪")

---
## 🏆 Zadanie bonusowe (opcjonalne): Prosty rate-limiter / flaga anomalii

**Trudność:** ⭐⭐⭐

Zaimplementuj prostą regułę wykrywania anomalii **wyłącznie operacjami RDD**: oznacz adres IP jako `PODEJRZANY`, jeśli wykonał **więcej niż 40 requestów w ciągu jednej godziny** (dowolnej godziny w naszym zbiorze).

**Sugerowane kroki:**
1. Zbuduj klucz złożony `(ip, godzina)` i zlicz requesty dla każdej takiej pary (`reduceByKey`)
2. Odfiltruj (`filter`) pary, gdzie liczba requestów > 40
3. Wypisz unikalne adresy IP oznaczone jako podejrzane (`map` + `distinct` lub `collect` + `set`)

Nie ma tu automatycznego testu — porównaj swój wynik z listą IP z Ćwiczenia 5. Czy to te same adresy? 🤔

In [ ]:
# 🏆 Bonus: rate-limiter / flaga anomalii (RDD)

# TODO: klucz złożony (ip, godzina) -> 1, zsumuj przez reduceByKey
ip_godzina_rdd = ...  # sparsowane_rdd.map(lambda r: ((r[0], r[1][11:13]), 1)).reduceByKey(lambda a, b: a + b)

# TODO: odfiltruj pary z liczbą requestów > 40
podejrzane_pary_rdd = ...  # ip_godzina_rdd.filter(lambda kv: kv[1] > 40)

# TODO: wyciągnij unikalne adresy IP z podejrzanych par
podejrzane_ip = ...  # podejrzane_pary_rdd.map(lambda kv: kv[0][0]).distinct().collect()

print(f"🚨 Podejrzane adresy IP (potencjalne boty): {podejrzane_ip}")

In [ ]:
# @title 💡 Pełne rozwiązanie — Bonus (kliknij, aby zobaczyć krok po kroku)

# Krok 1: budujemy klucz złożony (ip, godzina) i zliczamy requesty dla każdej takiej pary
ip_godzina_rdd = sparsowane_rdd.map(lambda r: ((r[0], r[1][11:13]), 1)).reduceByKey(lambda a, b: a + b)

# Krok 2: odfiltrowujemy tylko pary, w których jeden IP wykonał > 40 requestów w jednej godzinie
podejrzane_pary_rdd = ip_godzina_rdd.filter(lambda kv: kv[1] > 40)

# Krok 3: wyciągamy unikalne adresy IP z podejrzanych par
podejrzane_ip = podejrzane_pary_rdd.map(lambda kv: kv[0][0]).distinct().collect()

print(f"🚨 Podejrzane adresy IP (potencjalne boty): {podejrzane_ip}")


---
## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Parsowanie surowych linii tekstu na strukturę za pomocą `map`
- ✅ Filtrowanie danych (`filter`) — np. wyłącznie błędów HTTP
- ✅ Agregacje po kluczu (`reduceByKey`) — ruch na endpoint, error rate, requesty per IP
- ✅ Sortowanie i selekcję ekstremów (`takeOrdered`) — najwolniejsze requesty, top IP
- ✅ Wykrywanie anomalii wyłącznie operacjami RDD (bonus)

### 📚 Dalsza lektura
- [Apache Spark — RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html)
- [PySpark RDD API Reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.html#rdd-apis)

### 🔮 Co dalej?
Na **Wykładzie 5 (03.11.2026)** poznamy **Spark SQL i DataFrames** — te same operacje, ale ze schematem, optymalizatorem Catalyst i... zwykłym SQL! 🚀